## Install Dependencies

In [14]:
!pip install timm torchmetrics

## Imports

In [15]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms
import timm

from sklearn.metrics.pairwise import cosine_similarity

## Paths

In [16]:
TRAIN_DIR = "/kaggle/input/competitions/jaguar-re-id/train/train"
TEST_DIR = "/kaggle/input/competitions/jaguar-re-id/test/test"

TRAIN_CSV = "/kaggle/input/competitions/jaguar-re-id/train.csv"
TEST_CSV = "/kaggle/input/competitions/jaguar-re-id/test.csv"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## Load CSV

In [17]:
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print(train_df.head())
print(test_df.head())

         filename ground_truth
0  train_0001.png        Abril
1  train_0002.png        Abril
2  train_0003.png        Abril
3  train_0004.png       Akaloi
4  train_0005.png       Akaloi
   row_id    query_image  gallery_image
0       0  test_0001.png  test_0002.png
1       1  test_0001.png  test_0003.png
2       2  test_0001.png  test_0004.png
3       3  test_0001.png  test_0005.png
4       4  test_0001.png  test_0006.png


## Label Encoding

In [18]:
labels = train_df["ground_truth"].unique()
label_map = {l:i for i,l in enumerate(labels)}

train_df["label"] = train_df["ground_truth"].map(label_map)

NUM_CLASSES = len(label_map)

print("Number of Jaguars:", NUM_CLASSES)

Number of Jaguars: 31


## Image Augmentation

In [19]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2,contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

## Dataset Class

In [20]:
class JaguarDataset(Dataset):

    def __init__(self, df, img_dir, transform):

        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        img = Image.open(
            os.path.join(self.img_dir,row.filename)
        ).convert("RGB")

        img = self.transform(img)

        label = row.label

        return img,label

## DataLoader

In [21]:
train_dataset = JaguarDataset(
    train_df,
    TRAIN_DIR,
    train_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=4
)

## Model — Swin Transformer

In [22]:
class JaguarReID(nn.Module):

    def __init__(self,num_classes=31,embedding_dim=512):

        super().__init__()

        self.backbone = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=True,
            num_classes=0
        )

        in_features = self.backbone.num_features

        self.embedding = nn.Linear(in_features,embedding_dim)

        self.bn = nn.BatchNorm1d(embedding_dim)

        self.classifier = nn.Linear(embedding_dim,num_classes)

    def forward(self,x):

        feat = self.backbone(x)

        emb = self.embedding(feat)

        emb = self.bn(emb)

        logits = self.classifier(emb)

        return emb,logits

## Loss Functions
### ArcFace

In [23]:
class ArcFaceLoss(nn.Module):

    def __init__(self,s=30.0,m=0.5):
        super().__init__()
        self.s = s
        self.m = m

    def forward(self,logits,labels):

        one_hot = torch.zeros_like(logits)
        one_hot.scatter_(1,labels.view(-1,1),1)

        logits = logits - one_hot*self.m

        loss = F.cross_entropy(self.s*logits,labels)

        return loss

### Triplet Loss

In [24]:
triplet_loss = nn.TripletMarginLoss(margin=1.0)

## Training Setup

In [25]:
model = JaguarReID(NUM_CLASSES).to(DEVICE)

arcface = ArcFaceLoss()

optimizer = torch.optim.Adam(model.parameters(),lr=1e-4)

EPOCHS = 10

## Training Loop

In [26]:
for epoch in range(EPOCHS):

    model.train()
    total_loss = 0

    for imgs,labels in tqdm(train_loader):

        imgs = imgs.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        emb,logits = model(imgs)

        loss = arcface(logits,labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print("Epoch:",epoch,"Avg Loss:",avg_loss)

100%|██████████| 119/119 [04:39<00:00,  2.35s/it]


Epoch: 0 Avg Loss: 19.286750035125667


100%|██████████| 119/119 [04:44<00:00,  2.39s/it]


Epoch: 1 Avg Loss: 4.732457586832463


100%|██████████| 119/119 [04:38<00:00,  2.34s/it]


...

Epoch: 7 Avg Loss: 0.9906038191671127


100%|██████████| 119/119 [04:41<00:00,  2.37s/it]


Epoch: 8 Avg Loss: 0.8110550502292684


100%|██████████| 119/119 [04:46<00:00,  2.41s/it]

Epoch: 9 Avg Loss: 0.7949105773061358


## Save Model

In [27]:
torch.save(model.state_dict(),"jaguar_model.pth")

## Embedding Extraction

In [28]:
model.eval()

embedding_cache = {}

def get_embedding(img_name):

    if img_name in embedding_cache:
        return embedding_cache[img_name]

    img = Image.open(
        os.path.join(TEST_DIR,img_name)
    ).convert("RGB")

    img = test_transform(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():

        emb,_ = model(img)

    emb = emb.cpu().numpy()

    embedding_cache[img_name] = emb

    return emb

## Compute Similarity

In [29]:
similarities = []

for _,row in tqdm(test_df.iterrows(),total=len(test_df)):

    e1 = get_embedding(row.query_image)
    e2 = get_embedding(row.gallery_image)

    sim = cosine_similarity(e1,e2)[0][0]

    sim = (sim + 1) / 2

    similarities.append(sim)

100%|██████████| 137270/137270 [03:34<00:00, 639.14it/s] 


## Create Submission.csv

In [30]:
submission = pd.DataFrame({
    "row_id": test_df.row_id,
    "similarity": similarities
})

submission.to_csv("submission.csv",index=False)

print("Submission saved!")

Submission saved!


## Check First Rows

In [31]:
import pandas as pd

sub = pd.read_csv("submission.csv")
print(sub.head())

   row_id  similarity
0       0    0.842276
1       1    0.451832
2       2    0.528437
3       3    0.495221
4       4    0.543253


## Check for Errors (NaN values)

In [38]:
print(test_df["similarity"].isnull().sum())

0


## Check File Shape

In [32]:
print(len(test_df))

137270


## Add the Column similarities

In [34]:
test_df["similarity"] = similarities

## Final Validation Script

In [35]:
print("Rows:", len(test_df))
print("Min similarity:", test_df["similarity"].min())
print("Max similarity:", test_df["similarity"].max())
print("Missing values:", test_df["similarity"].isnull().sum())
print(test_df.head())

Rows: 137270
Min similarity: 0.20497268438339233
Max similarity: 0.998244047164917
Missing values: 0
   row_id    query_image  gallery_image  similarity
0       0  test_0001.png  test_0002.png    0.842276
1       1  test_0001.png  test_0003.png    0.451832
2       2  test_0001.png  test_0004.png    0.528437
3       3  test_0001.png  test_0005.png    0.495221
4       4  test_0001.png  test_0006.png    0.543253
